# Session 13.3: Cat and Dog Classification with ResNet18

**Learning goals:**

- Download labelled cat and dog images using KaggleHub.
- Prepare training and test batches for pretrained ResNet18.
- Freeze the feature extractor and train a two-class classification head.
- Evaluate predictions using accuracy and macro F1.

**Prerequisites:**

- Session 12: CNN training and transfer learning with PyTorch.
- Python with PyTorch, torchvision, Pillow, Matplotlib, scikit-learn and KaggleHub installed.
- Internet access for the dataset and pretrained weights on the first run.
- Save this notebook in the AI_SS course folder and run it from that folder.

This notebook contains the practical example from [Session 13, Section 4](<Session13 Image Classification and Computer Vision Applications.ipynb#4-case-study-dog-and-cat-classification-with-resnet18>).
Refer to [Session 12, Section 5](<Session12 Building and Training CNN Models with PyTorch.ipynb#5-transfer-learning-with-a-pretrained-cnn>) for the transfer-learning explanation. Run the cells in order.


## 1. Download the images

Install KaggleHub with `pip install kagglehub` in your terminal if needed. Run this notebook from the AI_SS course folder.
KaggleHub downloads the latest [Dog vs Cat dataset](https://www.kaggle.com/datasets/anthonytherrien/dog-vs-cat) and returns its local folder. The first run needs internet access; later runs reuse the download.

[KaggleHub documentation](https://github.com/Kaggle/kagglehub)

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("anthonytherrien/dog-vs-cat")

print("Path to dataset files:", path)

## 2. Prepare training and test batches

Find the folder containing the cat and dog class folders, then select up to 100 images per class for a short classroom run.
Split each class 80/20 for training and testing. Keep these splits fixed when comparing models.

`weights.transforms()` applies the resize, 224 × 224 centre crop and normalisation expected by pretrained ResNet18. These photographs already have three RGB channels.

[ImageFolder documentation](https://docs.pytorch.org/vision/stable/generated/torchvision.datasets.ImageFolder.html) · [ResNet18 weights and transforms](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.resnet18.html)

In [ ]:
import torch
from torchvision import datasets
from torchvision.models import ResNet18_Weights
from torch.utils.data import DataLoader, Subset

# Locate the class folders even when the download has a wrapper directory.
class_roots = []
for folder, subfolders, _ in os.walk(path):
    if {name.lower() for name in subfolders} in ({"cat", "dog"}, {"cats", "dogs"}):
        class_roots.append(folder)
if len(class_roots) != 1:
    raise ValueError("Expected one folder containing cat and dog class folders. Inspect: " + path)

SEED = 42
MAX_IMAGES_PER_CLASS = 100  # Limit the classroom example; increase to use more images.
BATCH_SIZE = 16  # Process a small group of images at each training step.
torch.manual_seed(SEED)  # Reproduce the split and the new head's initial weights.
weights = ResNet18_Weights.DEFAULT
image_dataset = datasets.ImageFolder(class_roots[0], transform=weights.transforms())
class_names = image_dataset.classes  # Use ImageFolder's class-to-index order.

train_indices, test_indices = [], []
split_generator = torch.Generator().manual_seed(SEED)
for target in range(len(class_names)):
    indices = [i for i, label in enumerate(image_dataset.targets) if label == target]
    order = torch.randperm(len(indices), generator=split_generator).tolist()
    selected = [indices[i] for i in order[:MAX_IMAGES_PER_CLASS]]
    if len(selected) < 5:
        raise ValueError("Provide at least five images per class.")
    split_at = int(0.8 * len(selected))  # Reserve 20% of each class for testing.
    train_indices.extend(selected[:split_at])
    test_indices.extend(selected[split_at:])

train_loader = DataLoader(
    Subset(image_dataset, train_indices), batch_size=BATCH_SIZE, shuffle=True,
    generator=torch.Generator().manual_seed(SEED),  # Reproduce the shuffled training order.
)
test_dataset = Subset(image_dataset, test_indices)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
print("Classes:", class_names)
print("Training images:", len(train_indices), "Test images:", len(test_indices))

## 3. Load ResNet18 and configure transfer learning

As in Session 12, replace `fc` with a new classification head, freeze the backbone and give the optimiser only the head's parameters.
The model returns two logits, one for each class. `CrossEntropyLoss` uses these logits directly.
The first run downloads the ImageNet weights.

In [ ]:
from torch import nn
from torchvision.models import resnet18

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

torch.hub.set_dir("../archive/AI_SS/torch")  # Store downloaded pretrained weights locally.
model = resnet18(weights=weights)  # Reuse features learned from ImageNet.
model.fc = nn.Linear(model.fc.in_features, len(class_names))  # Produce cat and dog logits.
model = model.to(device)  # Move parameters before constructing the optimiser.

for parameter in model.parameters():
    parameter.requires_grad = False  # Freeze every parameter first.
for parameter in model.fc.parameters():
    parameter.requires_grad = True  # Train only the replacement head.

LEARNING_RATE = 0.001
loss_function = nn.CrossEntropyLoss()  # Compare logits with integer class labels.
optimiser = torch.optim.Adam(model.fc.parameters(), lr=LEARNING_RATE)
print("Device:", device, "New head:", model.fc)

## 4. Train the new classification head

Keep `model.eval()` during head training so the frozen backbone's BatchNorm statistics stay fixed, then select `model.fc.train()` for the head.
Evaluation mode does not disable gradients: `loss.backward()` still calculates them for the trainable head.
Each epoch processes every training batch. The printed loss is the average over all training images.

In [ ]:
EPOCHS = 3  # Make three passes through the training images.
for epoch in range(EPOCHS):
    model.eval()  # Preserve the backbone's BatchNorm running statistics.
    model.fc.train()  # Select training behaviour for the new head.
    total_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimiser.zero_grad()  # Clear gradients from the previous batch.
        logits = model(images)  # Extract features and calculate two class scores.
        loss = loss_function(logits, labels)
        loss.backward()  # Calculate gradients for the head.
        optimiser.step()  # Update only the head's parameters.
        total_loss += loss.item() * labels.size(0)
    print(f"Epoch {epoch + 1}/{EPOCHS}: loss={total_loss / len(train_indices):.3f}")

## 5. Evaluate and inspect predictions

Evaluate on the held-out test images. Accuracy reports the proportion of correct predictions; macro F1 gives each class equal weight.
Show five test images with their true and predicted labels. Open the original RGB files for display because the model inputs have been normalised.

Use a separate validation split if you want to tune settings; keep the test set for evaluating the final model.

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

model.eval()  # Use fixed layer behaviour for testing.
test_targets, test_predictions = [], []
with torch.no_grad():  # Predict without recording gradients.
    for images, labels in test_loader:
        predictions = model(images.to(device)).argmax(dim=1).cpu()
        test_predictions.extend(predictions.tolist())
        test_targets.extend(labels.tolist())

print(f"Test accuracy: {accuracy_score(test_targets, test_predictions):.1%}")
print(f"Macro F1: {f1_score(test_targets, test_predictions, average='macro', zero_division=0):.3f}")

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

DISPLAY_COUNT = min(5, len(test_dataset))
figure, axes = plt.subplots(1, DISPLAY_COUNT, figsize=(12, 3), squeeze=False)
for position, axis in enumerate(axes.flat):
    image_index = test_indices[position]  # Match the fixed test-loader order.
    image_path, actual = image_dataset.samples[image_index]
    with Image.open(image_path) as image:
        axis.imshow(image.convert("RGB"))  # Display the original colour pixels.
    predicted = test_predictions[position]
    axis.set_title(f"True: {class_names[actual]}\nPred: {class_names[predicted]}")
    axis.axis("off")
plt.tight_layout()
plt.show()